<div align="center">

#  NLP → Database Schema Generator
### *Expert System Edition*

[![Python](https://img.shields.io/badge/Python-3.8%2B-blue?logo=python&logoColor=white)](https://python.org)
[![spaCy](https://img.shields.io/badge/spaCy-3.x-09A3D5?logo=spacy&logoColor=white)](https://spacy.io)
[![Experta](https://img.shields.io/badge/Experta-Expert%20System-8A2BE2)](https://github.com/nilp0inter/experta)
[![MySQL](https://img.shields.io/badge/MySQL-Ready-4479A1?logo=mysql&logoColor=white)](https://mysql.com)
[![License](https://img.shields.io/badge/License-MIT-green)](LICENSE)

> **Transform plain English text into a full relational database schema automatically.**
> Powered by Natural Language Processing (spaCy) + a Rule-Based Expert System (Experta).

---

| Feature | Description |
|---------|-------------|
| 🔍 **NLP Analysis** | Deep parsing with spaCy (POS tagging, dependency trees, NER) |
| 🧠 **Expert System** | 7 inference rules → entities, attributes, relationships |
| 🗄️ **SQL Generation** | Full DDL with FK, indexes, constraints, timestamps |
| 📊 **ER Diagram** | ASCII/Unicode diagram rendered in terminal |
| 📤 **Export** | `.sql` · `.json` · `.html` reports |
| 🐬 **MySQL Ready** | Direct execution against a live MySQL database |

</div>

## 📋 Table of Contents

1. [Installation & Setup](#1-installation--setup)
2. [Architecture Overview](#2-architecture-overview)
3. [Core Data Structures — Experta Facts](#3-core-data-structures--experta-facts)
4. [Expert System Engine — Inference Rules](#4-expert-system-engine--inference-rules)
5. [NLP Processor — spaCy Pipeline](#5-nlp-processor--spacy-pipeline)
6. [Schema Builder — DDL Generator](#6-schema-builder--ddl-generator)
7. [ER Diagram Renderer](#7-er-diagram-renderer)
8. [Export Engine (SQL · JSON · HTML)](#8-export-engine-sql--json--html)
9. [Database Executor — MySQL](#9-database-executor--mysql)
10. [📝 Interactive Demo — Enter Your Own Text!](#10-interactive-demo--enter-your-own-text)
11. [Results & Statistics](#11-results--statistics)

---
## 1. Installation & Setup

Install all required libraries and download the spaCy language model.

> ⚠️ Run this cell **once** before anything else.

In [7]:
# ── Install dependencies ────────────────────────────────────────────────────
import sys

PACKAGES = [
    "spacy",
    "experta",
    "rich",
    "mysql-connector-python",
]

for pkg in PACKAGES:
    print(f"Installing {pkg}...", end=" ")
    ret = __import__("subprocess").run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True
    )
    print("✓" if ret.returncode == 0 else f"✗ {ret.stderr.decode()[:60]}")

# Download spaCy model
print("\nDownloading spaCy English model...", end=" ")
ret = __import__("subprocess").run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"],
    capture_output=True
)
print("✓" if ret.returncode == 0 else "⚠ Already installed or failed")
print("\n✅ Setup complete!")

Installing spacy... ✓
Installing experta... ✓
Installing rich... ✓
Installing mysql-connector-python... ✓


✅ Setup complete!


---
## 2. Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    INPUT: Plain English Text                  │
└───────────────────────────┬─────────────────────────────────┘
                            │
                            ▼
┌─────────────────────────────────────────────────────────────┐
│                  NLP PROCESSOR  (spaCy)                       │
│  • Tokenisation  • POS Tagging  • Dependency Parsing         │
│  • Named Entity Recognition  • Lemmatisation                  │
└───────────────────────────┬─────────────────────────────────┘
                            │  TokenFacts + DependencyFacts
                            ▼
┌─────────────────────────────────────────────────────────────┐
│              EXPERT SYSTEM ENGINE  (Experta)                  │
│  Rule 1: Create Entity from Nouns                             │
│  Rule 2: Add Attributes via "have/include" verbs             │
│  Rule 3: Detect 1-to-Many  (singular → plural)              │
│  Rule 4: Detect Many-to-Many (plural → plural)               │
│  Rule 5: Auto-create Join Tables for M:M                     │
│  Rule 6: Semantic Enrichment (domain knowledge)              │
│  Rule 7: Ensure Primary Key on every entity                  │
└───────────────────────────┬─────────────────────────────────┘
                            │  EntityFacts + RelationshipFacts
                            ▼
┌─────────────────────────────────────────────────────────────┐
│               SCHEMA BUILDER  (DDL Generator)                 │
│  • Infer SQL column types  • Add FK constraints              │
│  • Create indexes  • Add timestamps  • Sort tables           │
└───────────────────────────┬─────────────────────────────────┘
                            │
              ┌─────────────┼───────────────┐
              ▼             ▼               ▼
        ER Diagram      SQL File      JSON + HTML
        (Terminal)     (.sql)         Reports

---
## 3. Core Imports & Global Configuration

All libraries, constants, and type mappings used across the entire system.

In [9]:
# ── Compatibility fix for older Python versions ──────────────────────────────
import collections
import collections.abc
collections.Mapping = collections.abc.Mapping

# ── Standard library ─────────────────────────────────────────────────────────
import sys, os, json, re, time, datetime
from collections import defaultdict
from typing import List, Dict, Optional, Tuple, Any

# ── NLP ──────────────────────────────────────────────────────────────────────
import spacy
from spacy.tokens import Token, Doc

# ── Expert System ─────────────────────────────────────────────────────────────
from experta import (
    KnowledgeEngine, Rule, Fact, Field,
    MATCH, TEST, AS, NOT, OR, AND, DefFacts, InitialFact
)

# ── Terminal UI ───────────────────────────────────────────────────────────────
from rich.console import Console
from rich.table   import Table
from rich.panel   import Panel
from rich.syntax  import Syntax
from rich.tree    import Tree
from rich         import box
from rich.columns import Columns
from rich.rule    import Rule as RichRule

# ── MySQL (optional) ──────────────────────────────────────────────────────────
try:
    import mysql.connector
    MYSQL_AVAILABLE = True
except ImportError:
    MYSQL_AVAILABLE = False

console = Console()

# ── Version info ─────────────────────────────────────────────────────────────
VERSION  = "2.0.0-expert"
APP_NAME = "NLP → DB Schema Generator"

print(f"✅ All imports successful!")
print(f"   spaCy    : {spacy.__version__}")
print(f"   MySQL    : {'available' if MYSQL_AVAILABLE else 'not installed (optional)'}")
print(f"   App      : {APP_NAME} v{VERSION}")

✅ All imports successful!
   spaCy    : 3.8.11
   MySQL    : available
   App      : NLP → DB Schema Generator v2.0.0-expert


---
## 3. Core Data Structures — Experta Facts

### 3.1 SQL Type Mapping & Semantic Profiles

The system uses two lookup tables to produce intelligent column types:

- **`SQL_TYPES`** — maps keyword patterns in attribute names → SQL data types
- **`SEMANTIC_MAP`** — domain-specific entity profiles with pre-known attributes

In [10]:
# ── SQL column type inference table ─────────────────────────────────────────
# Key = substring to look for in column name → Value = SQL type string
SQL_TYPES = {
    "id":          "INT UNSIGNED NOT NULL AUTO_INCREMENT",
    "name":        "VARCHAR(150) NOT NULL",
    "title":       "VARCHAR(200) NOT NULL",
    "description": "TEXT",
    "date":        "DATE",
    "email":       "VARCHAR(100) UNIQUE",
    "phone":       "VARCHAR(20)",
    "address":     "VARCHAR(300)",
    "number":      "VARCHAR(50) UNIQUE",
    "code":        "VARCHAR(50) UNIQUE NOT NULL",
    "status":      "ENUM('active','inactive','pending') DEFAULT 'active'",
    "count":       "INT DEFAULT 0",
    "score":       "FLOAT",
    "url":         "VARCHAR(500)",
    "default":     "VARCHAR(255)",
}

# ── Domain-specific entity enrichment map ───────────────────────────────────
# For known entity types the engine automatically adds these extra attributes
SEMANTIC_MAP = {
    "professor":   {"extra_attrs": ["full_name","email","expertise_area","hire_date","office_number"], "color": "cyan"},
    "student":     {"extra_attrs": ["full_name","email","enrollment_date","phone","address","gpa"],    "color": "green"},
    "subject":     {"extra_attrs": ["title","description","credits","semester","max_students"],        "color": "yellow"},
    "program":     {"extra_attrs": ["title","description","duration_years","degree_level"],            "color": "magenta"},
    "faculty":     {"extra_attrs": ["name","dean_name","established_year","location"],                 "color": "blue"},
    "laboratory":  {"extra_attrs": ["name","location","capacity","equipment_type","supervisor_id"],    "color": "red"},
    "thesis":      {"extra_attrs": ["title","abstract","submission_date","status","supervisor_id"],    "color": "bright_cyan"},
    "publication": {"extra_attrs": ["title","author_name","publish_date","journal_name","doi"],        "color": "bright_green"},
    "exam":        {"extra_attrs": ["title","exam_date","duration_minutes","max_score"],               "color": "bright_yellow"},
    "assignment":  {"extra_attrs": ["title","due_date","max_score","description"],                    "color": "bright_magenta"},
    "campus":      {"extra_attrs": ["name","address","area_sqm"],                                     "color": "bright_blue"},
    "university":  {"extra_attrs": ["name","address","founded_year","website"],                       "color": "white"},
}

# ── Stop words (too generic to be entities) ──────────────────────────────────
STOP_WORDS = {
    "the","a","an","each","every","some","any","all",
    "this","that","these","those","its","their",
    "they","it","he","she","we","you","i",
    "university","course",
}

print(f"✅ Configured {len(SQL_TYPES)} SQL type mappings")
print(f"   Known entity profiles : {len(SEMANTIC_MAP)}")
print(f"   Stop words            : {len(STOP_WORDS)}")

✅ Configured 15 SQL type mappings
   Known entity profiles : 12
   Stop words            : 23


### 3.2 Experta Fact Classes

Facts are the **working memory** of the expert system.
Each fact class represents a piece of knowledge that rules can match against and produce.

| Fact Class | Purpose |
|------------|---------|
| `TokenFact` | One spaCy token (noun/proper noun) with its grammatical properties |
| `DependencyFact` | A subject→verb→object triple extracted from the dependency parse |
| `EntityFact` | A discovered database table with its attributes |
| `RelationshipFact` | A typed relationship (1:1, 1:M, M:M) between two entities |
| `AttributeFact` | A single column attached to an entity |
| `PrerequisiteFact` | A prerequisite link between two subjects |

In [11]:
class TokenFact(Fact):
    """
    Represents a single spaCy token that is a NOUN or PROPN.
    Stores grammatical metadata used by inference rules.
    """
    token        = Field(object, mandatory=True)   # spaCy Token object
    lemma        = Field(str,    mandatory=True)   # base form of the word
    pos          = Field(str,    mandatory=True)   # NOUN / PROPN
    dep          = Field(str,    mandatory=True)   # nsubj / dobj / ROOT …
    tag          = Field(str,    mandatory=True)   # NN / NNS / NNP / NNPS
    is_plural    = Field(bool,   default=False)    # True for NNS / NNPS
    is_proper    = Field(bool,   default=False)    # True for proper nouns


class DependencyFact(Fact):
    """
    Syntactic dependency triple: subject → verb → object.
    Used to detect relationships and attributes between entities.
    """
    subject       = Field(object, mandatory=True)
    subject_lemma = Field(str,    mandatory=True)
    verb          = Field(str,    mandatory=True)   # lemma of the verb
    object_       = Field(object, mandatory=True)
    object_lemma  = Field(str,    mandatory=True)
    sentence_idx  = Field(int,    default=0)        # which sentence


class EntityFact(Fact):
    """
    A discovered database entity that will become a table.
    Attributes accumulate as more rules fire.
    """
    name          = Field(str,   mandatory=True)
    attributes    = Field(tuple, default=lambda: ())
    source_token  = Field(str,   default="")
    confidence    = Field(float, default=1.0)
    is_join_table = Field(bool,  default=False)


class RelationshipFact(Fact):
    """
    A typed relationship between two entities.
    rel_type ∈ { '1:1', '1:m', 'm:1', 'm:m' }
    """
    entity1   = Field(str, mandatory=True)
    entity2   = Field(str, mandatory=True)
    rel_type  = Field(str, mandatory=True)
    verb      = Field(str, default="related_to")
    fk_column = Field(str, default="")


class AttributeFact(Fact):
    """An individual column discovered for a specific entity."""
    entity_name = Field(str, mandatory=True)
    attr_name   = Field(str, mandatory=True)
    attr_type   = Field(str, default="VARCHAR(255)")


class PrerequisiteFact(Fact):
    """A prerequisite dependency between two subjects/courses."""
    subject  = Field(str, mandatory=True)
    requires = Field(str, mandatory=True)


print("✅ All 6 Experta Fact classes defined.")

✅ All 6 Experta Fact classes defined.


---
## 4. Expert System Engine — Inference Rules

The `UniversityDBEngine` is the **brain** of the system.
It inherits from `KnowledgeEngine` and declares 7 rules.

### How Experta works
1. You **declare facts** into the working memory.
2. The engine checks every rule's **conditions** (left-hand side).
3. When all conditions match, the rule **fires** (right-hand side executes).
4. Newly declared facts can trigger further rules → **forward chaining**.

### Rule Summary

| # | Rule | Trigger | Action |
|---|------|---------|--------|
| 1 | `rule_create_entity_from_noun` | Any NOUN/PROPN token not in stop-words | Declare `EntityFact` |
| 2 | `rule_add_attribute_via_have` | verb ∈ {have, include, contain, feature} | Add attribute to matching entity |
| 3 | `rule_one_to_many` | singular subject → plural object | Declare `1:m` relationship |
| 4 | `rule_many_to_many` | plural subject → plural object | Declare `m:m` relationship |
| 5 | `rule_create_join_table` | Any `m:m` relationship | Auto-create junction table |
| 6 | `rule_semantic_enrichment` | Entity name ∈ SEMANTIC_MAP | Enrich with domain attributes |
| 7 | `rule_ensure_primary_key` | Entity without `id` in attributes | Prepend `id` column |

In [12]:
class UniversityDBEngine(KnowledgeEngine):
    """
    Expert System Engine for NLP-driven Database Schema Inference.

    Rules fire via forward-chaining whenever their conditions are satisfied.
    The order of rule firing is non-deterministic; rules are designed to be
    idempotent (safe to fire multiple times via existence checks).
    """

    # ── Private helpers ───────────────────────────────────────────────────────

    def _entity_exists(self, name: str) -> bool:
        """Check if an EntityFact with this name already exists in WM."""
        return any(
            isinstance(f, EntityFact) and f["name"] == name
            for f in self.facts.values()
        )

    def _relationship_exists(self, e1: str, e2: str, rel: str) -> bool:
        """Check if a RelationshipFact (e1, e2, rel_type) already exists."""
        return any(
            isinstance(f, RelationshipFact)
            and f["entity1"] == e1 and f["entity2"] == e2 and f["rel_type"] == rel
            for f in self.facts.values()
        )

    def _is_meaningful(self, lemma: str) -> bool:
        """Return True if the lemma is worth adding as an attribute."""
        return len(lemma) > 2 and lemma.lower() not in STOP_WORDS and not lemma.isdigit()

    # ── Rule 1 ────────────────────────────────────────────────────────────────
    @Rule(
        TokenFact(lemma=MATCH.lemma, pos=MATCH.pos),
        TEST(lambda pos:   pos   in {"NOUN", "PROPN"}),
        TEST(lambda lemma: lemma not in STOP_WORDS and len(lemma) > 2)
    )
    def rule_create_entity_from_noun(self, lemma):
        """
        RULE 1 — Entity Discovery
        ─────────────────────────
        Every meaningful noun in the text is a candidate database table.
        Fires for every TokenFact whose POS is NOUN or PROPN and whose
        lemma is not a stop-word.
        """
        name = lemma.capitalize()
        if not self._entity_exists(name):
            self.declare(EntityFact(name=name, source_token=lemma))

    # ── Rule 2 ────────────────────────────────────────────────────────────────
    @Rule(
        DependencyFact(subject_lemma=MATCH.subj, verb=MATCH.verb, object_lemma=MATCH.obj),
        TEST(lambda verb: verb in {"have", "include", "contain", "feature", "possess"}),
        AS.entity << EntityFact(name=MATCH.ename, attributes=MATCH.attrs),
        TEST(lambda ename, subj: ename.lower() == subj.lower())
    )
    def rule_add_attribute_via_have(self, subj, obj, entity, attrs, ename):
        """
        RULE 2 — Attribute Extraction
        ──────────────────────────────
        When a sentence says 'Entity has/includes X', X becomes an attribute
        of that entity.  Example: 'Every professor has a full name and email'
        adds [full_name, email] to the Professor entity.
        """
        if obj not in attrs and self._is_meaningful(obj):
            self.modify(entity, attributes=attrs + (obj,))

    # ── Rule 3 ────────────────────────────────────────────────────────────────
    @Rule(
        DependencyFact(
            subject=MATCH.subj_tok, subject_lemma=MATCH.subj,
            verb=MATCH.verb,
            object_=MATCH.obj_tok,  object_lemma=MATCH.obj
        ),
        EntityFact(name=MATCH.e1),
        EntityFact(name=MATCH.e2),
        TEST(lambda e1, subj: e1.lower() == subj.lower()),
        TEST(lambda e2, obj:  e2.lower() == obj.lower()),
        TEST(lambda e1, e2:   e1 != e2),
        TEST(lambda subj_tok, obj_tok:
             subj_tok.tag_ not in {"NNS","NNPS"} and obj_tok.tag_ in {"NNS","NNPS"})
    )
    def rule_one_to_many(self, subj, obj, e1, e2, verb):
        """
        RULE 3 — One-to-Many Relationship
        ───────────────────────────────────
        Singular subject acting on a plural object implies 1:M.
        Example: 'A faculty offers several programs'
          → Faculty (singular) → Program (plural) = 1:M
        """
        if not self._relationship_exists(e1, e2, "1:m"):
            self.declare(RelationshipFact(entity1=e1, entity2=e2, rel_type="1:m", verb=verb))

    # ── Rule 4 ────────────────────────────────────────────────────────────────
    @Rule(
        DependencyFact(
            subject=MATCH.subj_tok, subject_lemma=MATCH.subj,
            verb=MATCH.verb,
            object_=MATCH.obj_tok,  object_lemma=MATCH.obj
        ),
        EntityFact(name=MATCH.e1),
        EntityFact(name=MATCH.e2),
        TEST(lambda e1, subj: e1.lower() == subj.lower()),
        TEST(lambda e2, obj:  e2.lower() == obj.lower()),
        TEST(lambda e1, e2:   e1 != e2),
        TEST(lambda subj_tok, obj_tok:
             subj_tok.tag_ in {"NNS","NNPS"} and obj_tok.tag_ in {"NNS","NNPS"})
    )
    def rule_many_to_many(self, subj, obj, e1, e2, verb):
        """
        RULE 4 — Many-to-Many Relationship
        ────────────────────────────────────
        Both subject and object are plural → M:M relationship.
        Example: 'Students register for multiple subjects'
          → Students (plural) ↔ Subjects (plural) = M:M
        """
        if not self._relationship_exists(e1, e2, "m:m"):
            self.declare(RelationshipFact(entity1=e1, entity2=e2, rel_type="m:m", verb=verb))

    # ── Rule 5 ────────────────────────────────────────────────────────────────
    @Rule(RelationshipFact(entity1=MATCH.e1, entity2=MATCH.e2, rel_type="m:m"))
    def rule_create_join_table(self, e1, e2):
        """
        RULE 5 — Junction Table Creation
        ──────────────────────────────────
        Every M:M relationship requires a junction (join) table.
        Automatically creates  E1_E2  with FK columns to both parents
        and replaces the M:M with two 1:M relationships.
        """
        join_name = f"{e1}_{e2}"
        if not self._entity_exists(join_name):
            self.declare(EntityFact(
                name=join_name,
                attributes=("id", f"id_{e1.lower()}", f"id_{e2.lower()}", "created_at"),
                is_join_table=True, confidence=0.95
            ))
            if not self._relationship_exists(e1, join_name, "1:m"):
                self.declare(RelationshipFact(
                    entity1=e1, entity2=join_name, rel_type="1:m",
                    verb="links", fk_column=f"id_{e1.lower()}"
                ))
            if not self._relationship_exists(e2, join_name, "1:m"):
                self.declare(RelationshipFact(
                    entity1=e2, entity2=join_name, rel_type="1:m",
                    verb="links", fk_column=f"id_{e2.lower()}"
                ))

    # ── Rule 6 ────────────────────────────────────────────────────────────────
    @Rule(
        AS.entity << EntityFact(name=MATCH.name, attributes=MATCH.attrs),
        TEST(lambda name: name.lower() in SEMANTIC_MAP)
    )
    def rule_semantic_enrichment(self, entity, name, attrs):
        """
        RULE 6 — Semantic Domain Enrichment
        ─────────────────────────────────────
        When an entity matches a known domain concept (student, professor…)
        automatically add its well-known attributes from SEMANTIC_MAP.
        This injects expert domain knowledge into the schema.
        """
        extra     = tuple(SEMANTIC_MAP[name.lower()]["extra_attrs"])
        new_attrs = tuple(dict.fromkeys(("id",) + extra + attrs))
        if new_attrs != attrs:
            self.modify(entity, attributes=new_attrs, confidence=1.0)

    # ── Rule 7 ────────────────────────────────────────────────────────────────
    @Rule(
        AS.entity << EntityFact(name=MATCH.name, attributes=MATCH.attrs),
        TEST(lambda attrs: "id" not in attrs)
    )
    def rule_ensure_primary_key(self, entity, attrs):
        """
        RULE 7 — Primary Key Guarantee
        ────────────────────────────────
        Every table must have a primary key.
        If an entity has no 'id' column, prepend one automatically.
        """
        self.modify(entity, attributes=("id",) + attrs)


print("✅ UniversityDBEngine defined with 7 inference rules.")

✅ UniversityDBEngine defined with 7 inference rules.


---
## 5. NLP Processor — spaCy Pipeline

The `NLPProcessor` class wraps spaCy and converts a raw text string into
structured `TokenFact` and `DependencyFact` objects ready for the expert system.

### What it does
- **Tokenisation** — splits text into words/punctuation
- **POS Tagging** — labels each word: NOUN, VERB, ADJ, …
- **Dependency Parsing** — finds grammatical relations: nsubj, dobj, ROOT
- **Lemmatisation** — reduces words to base form (professors → professor)
- **NER** — identifies named entities (person names, organisations, …)

In [14]:
class NLPProcessor:
    """
    Wraps a spaCy pipeline and extracts structured facts
    that the expert system engine can reason about.
    """

    def __init__(self, model_name: str = "en_core_web_sm"):
        """Load the spaCy language model."""
        print(f"Loading spaCy model '{model_name}'...", end=" ")
        self.nlp = spacy.load(model_name)
        print("✓")

    def process(self, text: str) -> Doc:
        """Run the full spaCy pipeline on the input text."""
        return self.nlp(text)

    def extract_facts(self, doc: Doc) -> list:
        """
        Walk every sentence and token.
        Produce:
          • TokenFact   — for every meaningful NOUN / PROPN
          • DependencyFact — for every VERB that has both a subject and object
        """
        facts       = []
        seen_tokens = set()

        for sent_idx, sent in enumerate(doc.sents):

            # ── TokenFacts ──────────────────────────────────────────────────
            for token in sent:
                lemma = token.lemma_.lower()
                if token.pos_ in {"NOUN","PROPN"} and lemma not in seen_tokens:
                    seen_tokens.add(lemma)
                    facts.append(TokenFact(
                        token     = token,
                        lemma     = lemma,
                        pos       = token.pos_,
                        dep       = token.dep_,
                        tag       = token.tag_,
                        is_plural = token.tag_ in {"NNS","NNPS"},
                        is_proper = token.pos_ == "PROPN",
                    ))

            # ── DependencyFacts ─────────────────────────────────────────────
            for token in sent:
                if token.pos_ == "VERB":
                    subj = next(
                        (c for c in token.children if c.dep_ in {"nsubj","nsubjpass"}), None
                    )
                    obj_ = next(
                        (c for c in token.children if c.dep_ in {"dobj","attr","nsubjpass"}), None
                    )
                    if subj and obj_:
                        facts.append(DependencyFact(
                            subject       = subj,
                            subject_lemma = subj.lemma_.lower(),
                            verb          = token.lemma_,
                            object_       = obj_,
                            object_lemma  = obj_.lemma_.lower(),
                            sentence_idx  = sent_idx,
                        ))
        return facts

    def show_dependency_tree(self, doc: Doc):
        """Render a colour-coded dependency tree using Rich."""
        tree = Tree("[bold yellow]🌳 Dependency Parse Tree[/bold yellow]")
        for sent in doc.sents:
            # الحل: استخدام علامات اقتباس مفردة للنص الداخلي أو استخدام علامات اقتباس مزدوجة مع الهروب
            sent_text = sent.text[:70] + ('...' if len(sent.text) > 70 else '')
            branch = tree.add(
                f'[dim italic]"{sent_text}"[/dim italic]'
            )
            for token in sent:
                if token.dep_ in {"nsubj","dobj","ROOT","nsubjpass"}:
                    color = {"ROOT":"red","nsubj":"green","dobj":"cyan","nsubjpass":"magenta"}.get(token.dep_,"white")
                    branch.add(
                        f"[{color}]{token.dep_:12s}[/{color}] "
                        f"[bold]{token.text}[/bold]  [dim]({token.pos_} | {token.tag_})[/dim]"
                    )
        console.print(tree)


print("✅ NLPProcessor class defined.")

✅ NLPProcessor class defined.


---
## 6. Schema Builder — DDL Generator

`SchemaBuilder` takes the lists of entities and relationships produced by the
expert system and converts them into a complete, production-ready SQL DDL script.

### Features
- Infers SQL column types by substring matching on attribute names
- Adds `FOREIGN KEY` constraints from detected relationships
- Creates `INDEX` and `UNIQUE INDEX` definitions automatically
- Appends `created_at` / `updated_at` timestamp columns to every table
- Sorts tables so join tables appear last (respecting FK order)
- Outputs MySQL-compatible DDL with InnoDB + utf8mb4

In [15]:
class SchemaBuilder:
    """
    Converts EntityFacts + RelationshipFacts into a complete SQL DDL script.
    Handles column type inference, FK constraints, indexes, and timestamps.
    """

    def __init__(self, entities: list, relationships: list):
        self.entities      = entities
        self.relationships = relationships
        self.schema: dict  = {}
        self._build()

    # ── Type inference ────────────────────────────────────────────────────────
    def _infer_sql_type(self, attr: str) -> str:
        """Map an attribute name to its most appropriate SQL data type."""
        attr_low = attr.lower()
        for key, sql_type in SQL_TYPES.items():
            if key in attr_low:
                return sql_type
        if attr_low.endswith("_id") or attr_low.startswith("id_"):
            return "INT UNSIGNED NOT NULL"
        if attr_low.endswith("_at") or attr_low.endswith("_date"):
            return "DATETIME DEFAULT CURRENT_TIMESTAMP"
        if attr_low in ("year","age","count","score","gpa","credits"):
            return "FLOAT"
        if attr_low in ("active","enabled","verified"):
            return "BOOLEAN DEFAULT TRUE"
        return "VARCHAR(255)"

    # ── Build internal schema dict ────────────────────────────────────────────
    def _build(self):
        """Populate self.schema from the entity and relationship lists."""
        entity_names = {e["name"] for e in self.entities}

        for entity in self.entities:
            name    = entity["name"]
            attrs   = entity.get("attributes", ("id",))
            is_join = entity.get("is_join_table", False)
            columns = {}

            # ── Column definitions ──────────────────────────────────────────
            for attr in attrs:
                if attr.lower() == "id":
                    columns[attr] = {
                        "type": "INT UNSIGNED NOT NULL AUTO_INCREMENT",
                        "primary_key": True, "nullable": False,
                    }
                else:
                    columns[attr] = {
                        "type": self._infer_sql_type(attr),
                        "primary_key": False,
                        "nullable": not attr.lower().endswith("_id"),
                    }

            # ── Foreign key columns ─────────────────────────────────────────
            fk_constraints = []
            for rel in self.relationships:
                if rel["entity2"] == name and rel["entity1"] in entity_names:
                    fk_col = f"id_{rel['entity1'].lower()}"
                    if fk_col not in columns:
                        columns[fk_col] = {"type": "INT UNSIGNED NOT NULL",
                                           "primary_key": False, "nullable": False}
                    fk_constraints.append({
                        "column": fk_col,
                        "references_table":  rel["entity1"],
                        "references_column": "id",
                        "on_delete": "CASCADE" if is_join else "RESTRICT",
                        "on_update": "CASCADE",
                    })

            # ── Auto timestamps ─────────────────────────────────────────────
            for ts_col, ts_type in [
                ("created_at", "DATETIME DEFAULT CURRENT_TIMESTAMP"),
                ("updated_at", "DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP"),
            ]:
                if ts_col not in columns:
                    columns[ts_col] = {"type": ts_type, "primary_key": False, "nullable": True}

            # ── Auto indexes ────────────────────────────────────────────────
            indexes = []
            for col_name, col_def in columns.items():
                if (col_name.endswith("_id")
                        or "UNIQUE" in col_def["type"]
                        or col_name in ("email","code","number")):
                    idx_type = "UNIQUE" if "UNIQUE" in col_def["type"] else "INDEX"
                    indexes.append({
                        "name": f"idx_{name.lower()}_{col_name}",
                        "columns": [col_name], "type": idx_type,
                    })

            self.schema[name] = {
                "columns": columns, "fk_constraints": fk_constraints,
                "indexes": indexes, "is_join_table": is_join,
            }

    # ── DDL generation ────────────────────────────────────────────────────────
    def generate_ddl(self, database_name: str = "university_db") -> str:
        """Produce a full MySQL DDL script as a string."""
        lines = []
        lines.append(
            f"-- ═══════════════════════════════════════════════════════\n"
            f"-- {APP_NAME} v{VERSION}\n"
            f"-- Generated : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
            f"-- Database  : {database_name}\n"
            f"-- ═══════════════════════════════════════════════════════\n\n"
            f"SET FOREIGN_KEY_CHECKS = 0;\n"
            f"SET SQL_MODE = 'STRICT_TRANS_TABLES,NO_ZERO_IN_DATE';\n\n"
            f"CREATE DATABASE IF NOT EXISTS `{database_name}`\n"
            f"  CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;\n\n"
            f"USE `{database_name}`;\n\n"
        )

        sorted_tables = sorted(self.schema.items(), key=lambda x: (x[1]["is_join_table"], x[0]))

        for table_name, table_def in sorted_tables:
            cols_sql, unique_cols, fk_sql = [], [], []

            for col_name, col_def in table_def["columns"].items():
                col_type = col_def["type"]
                if "UNIQUE" in col_type:
                    unique_cols.append(col_name)
                    col_type = col_type.replace("UNIQUE", "").strip()
                if col_def["primary_key"]:
                    cols_sql.append(f"  `{col_name}` {col_type}")
                elif not col_def["nullable"]:
                    cols_sql.append(f"  `{col_name}` {col_type}")
                else:
                    cols_sql.append(f"  `{col_name}` {col_type.replace('NOT NULL','').strip()}")

            pk = next((c for c, d in table_def["columns"].items() if d["primary_key"]), None)
            if pk:
                cols_sql.append(f"  PRIMARY KEY (`{pk}`)")
            for uc in unique_cols:
                cols_sql.append(f"  UNIQUE KEY `uq_{table_name.lower()}_{uc}` (`{uc}`)")
            for fk in table_def["fk_constraints"]:
                fk_sql.append(
                    f"  CONSTRAINT `fk_{table_name.lower()}_{fk['column']}`\n"
                    f"    FOREIGN KEY (`{fk['column']}`)\n"
                    f"    REFERENCES `{fk['references_table']}` (`{fk['references_column']}`)\n"
                    f"    ON DELETE {fk['on_delete']} ON UPDATE {fk['on_update']}"
                )

            comment = "-- Junction Table" if table_def["is_join_table"] else ""
            table_sql = (
                f"-- ─── {table_name} ──── {comment}\n"
                f"DROP TABLE IF EXISTS `{table_name}`;\n"
                f"CREATE TABLE `{table_name}` (\n"
                + ",\n".join(cols_sql + fk_sql)
                + f"\n) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;\n\n"
            )
            lines.append(table_sql)

            for idx in table_def["indexes"]:
                idx_type  = "UNIQUE INDEX" if idx["type"] == "UNIQUE" else "INDEX"
                cols_list = ", ".join(f"`{c}`" for c in idx["columns"])
                lines.append(f"CREATE {idx_type} `{idx['name']}` ON `{table_name}` ({cols_list});\n")
            lines.append("\n")

        lines.append("SET FOREIGN_KEY_CHECKS = 1;\n")
        lines.append(f"\n-- ✅ Done. Tables generated: {len(self.schema)}\n")
        return "".join(lines)


print("✅ SchemaBuilder class defined.")

✅ SchemaBuilder class defined.


---
## 7. ER Diagram Renderer

Draws a visual Entity-Relationship diagram directly in the terminal
using Rich tables, panels, and columns.

Each table is displayed as a card showing:
- Column names
- Inferred SQL types
- Key indicators (🔑 PK, 🔗 FK, ◆ Unique)

In [16]:
class ERDiagramRenderer:
    """
    Renders an ASCII/Unicode ER Diagram in the terminal.
    Uses Rich Panels arranged in a 3-column grid layout.
    """

    def __init__(self, schema: dict, relationships: list):
        self.schema        = schema
        self.relationships = relationships

    def render(self):
        """Print the full ER diagram: table cards + relationship table."""
        console.print()
        console.rule("[bold yellow] Entity-Relationship Diagram[/bold yellow]")
        console.print()

        panels = []
        for table_name, table_def in sorted(self.schema.items()):
            color = "cyan" if table_def["is_join_table"] else (
                SEMANTIC_MAP.get(table_name.lower(), {}).get("color", "white")
            )
            t = Table(show_header=True, header_style=f"bold {color}",
                      box=box.SIMPLE_HEAVY, min_width=30, border_style="dim")
            t.add_column("Column", style="bold", no_wrap=True)
            t.add_column("Type",   style="dim italic", no_wrap=True)
            t.add_column("Key",    justify="center", no_wrap=True)

            for col_name, col_def in table_def["columns"].items():
                key_icon = (
                    "[yellow]🔑[/yellow]" if col_def["primary_key"] else
                    "[blue]🔗[/blue]"    if col_name.endswith("_id") else
                    "[green]◆[/green]"   if "UNIQUE" in col_def["type"] else ""
                )
                ts = re.sub(r"NOT NULL|AUTO_INCREMENT|DEFAULT.*", "", col_def["type"]).strip()
                t.add_row(col_name, ts[:20] + ".." if len(ts) > 20 else ts, key_icon)

            label = "🔀 Join" if table_def["is_join_table"] else f"📦 {table_name}"
            panels.append(Panel(t, title=f"[bold {color}]{label}[/bold {color}]", border_style=color))

        for i in range(0, len(panels), 3):
            console.print(Columns(panels[i:i+3]))

        # ── Relationships table ───────────────────────────────────────────────
        console.print()
        console.rule("[bold magenta]🔗  Relationships[/bold magenta]")
        rel_tbl = Table(show_header=True, header_style="bold magenta",
                        box=box.ROUNDED, show_lines=True)
        rel_tbl.add_column("From",        style="bold cyan",  min_width=18)
        rel_tbl.add_column("Cardinality", justify="center",   min_width=12)
        rel_tbl.add_column("To",          style="bold green", min_width=18)
        rel_tbl.add_column("Verb",        style="dim",        min_width=14)

        icons  = {"1:1":"── 1:1 ──","1:m":"── 1:M ──","m:m":"── M:M ──"}
        colors = {"1:1":"yellow","1:m":"blue","m:m":"red"}

        for rel in self.relationships:
            rt = rel["rel_type"]
            ci = f"[{colors.get(rt,'white')}]{icons.get(rt,rt)}[/{colors.get(rt,'white')}]"
            rel_tbl.add_row(rel["entity1"], ci, rel["entity2"], rel.get("verb","—"))

        console.print(rel_tbl)


print("✅ ERDiagramRenderer class defined.")

✅ ERDiagramRenderer class defined.


---
## 8. Export Engine (SQL · JSON · HTML)

`ReportExporter` saves the generated schema in three formats:

| Format | Contents |
|--------|----------|
| `.sql` | Complete DDL script ready to run in MySQL |
| `.json` | Machine-readable schema with metadata & statistics |
| `.html` | Beautiful dark-themed web report viewable in any browser |

In [17]:
class ReportExporter:
    """
    Exports the generated schema to .sql, .json and .html files.
    All files share a timestamp-based name for easy organisation.
    """

    def __init__(self, schema: dict, relationships: list, ddl: str, database: str):
        self.schema        = schema
        self.relationships = relationships
        self.ddl           = ddl
        self.database      = database
        self.ts            = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    def export_sql(self, path: str = ".") -> str:
        fname = f"{path}/schema_{self.ts}.sql"
        with open(fname, "w", encoding="utf-8") as f:
            f.write(self.ddl)
        return fname

    def export_json(self, path: str = ".") -> str:
        fname = f"{path}/schema_{self.ts}.json"
        data  = {
            "meta": {
                "generated_at": self.ts, "database": self.database,
                "version": VERSION, "generator": APP_NAME,
            },
            "statistics": {
                "total_tables":         len(self.schema),
                "total_relationships":  len(self.relationships),
                "join_tables": sum(1 for v in self.schema.values() if v["is_join_table"]),
            },
            "tables": {
                name: {
                    "columns": {col: {"type":d["type"],"primary_key":d["primary_key"],"nullable":d["nullable"]}
                                for col,d in tdef["columns"].items()},
                    "foreign_keys":  tdef["fk_constraints"],
                    "indexes":       tdef["indexes"],
                    "is_join_table": tdef["is_join_table"],
                }
                for name, tdef in self.schema.items()
            },
            "relationships": [
                {"from":r["entity1"],"to":r["entity2"],"type":r["rel_type"],"verb":r.get("verb","")}
                for r in self.relationships
            ]
        }
        with open(fname, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        return fname

    def export_html(self, path: str = ".") -> str:
        fname    = f"{path}/schema_{self.ts}.html"
        tbl_html = ""
        for name, tdef in sorted(self.schema.items()):
            badge = ('<span style="background:#7c3aed;color:#fff;padding:1px 6px;'
                     'border-radius:4px;font-size:11px">JOIN</span>') if tdef["is_join_table"] else ""
            tbl_html += (
                f'<div class="tcard"><div class="thead">{name} {badge}</div>'
                f'<table><tr><th>Column</th><th>Type</th><th>Key</th></tr>'
            )
            for col, cdef in tdef["columns"].items():
                pk = "🔑" if cdef["primary_key"] else ("🔗" if col.endswith("_id") else "")
                tbl_html += f"<tr><td>{col}</td><td><code>{cdef['type'][:38]}</code></td><td>{pk}</td></tr>"
            tbl_html += "</table></div>"

        rel_html = "".join(
            f"<tr><td>{r['entity1']}</td><td><b>{r['rel_type']}</b></td>"
            f"<td>{r['entity2']}</td><td>{r.get('verb','')}</td></tr>"
            for r in self.relationships
        )

        html = f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8">
<title>{APP_NAME}</title>
<link href="https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Syne:wght@800&display=swap" rel="stylesheet">
<style>
  *{{box-sizing:border-box;margin:0;padding:0}}
  body{{background:#0a0e1a;color:#e2e8f0;font-family:'JetBrains Mono',monospace;padding:2rem}}
  h1{{font-family:'Syne',sans-serif;font-size:2rem;font-weight:800;
      background:linear-gradient(135deg,#06b6d4,#818cf8);
      -webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-bottom:.3rem}}
  .meta{{color:#6b7280;font-size:.8rem;margin-bottom:2rem}}
  .grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(300px,1fr));gap:1rem;margin-bottom:2rem}}
  .tcard{{background:#1e293b;border-radius:8px;overflow:hidden;border:1px solid #334155}}
  .thead{{background:#334155;padding:.6rem 1rem;font-weight:700;color:#f8fafc}}
  table{{width:100%;border-collapse:collapse}}
  th,td{{padding:.35rem .8rem;text-align:left;font-size:.78rem;border-bottom:1px solid #0f172a}}
  th{{background:#0f172a;color:#94a3b8}} td:first-child{{color:#7dd3fc}}
  code{{background:#0f172a;padding:1px 5px;border-radius:3px;font-size:.72rem;color:#86efac}}
  .rtable{{width:100%;border-collapse:collapse;background:#1e293b;border-radius:8px;overflow:hidden}}
  .rtable th{{background:#334155;padding:.5rem 1rem;color:#94a3b8}}
  .rtable td{{padding:.4rem 1rem;border-bottom:1px solid #334155;font-size:.82rem}}
  pre{{background:#1e293b;padding:1rem;border-radius:8px;font-size:.72rem;
      overflow:auto;color:#86efac;max-height:500px;margin-top:2rem}}
  h2{{color:#94a3b8;font-size:1.1rem;margin:1.5rem 0 .8rem}}
</style></head><body>
<h1>🗄️ {APP_NAME}</h1>
<p class="meta">Database: <b>{self.database}</b> · Generated: {self.ts} · Tables: {len(self.schema)}</p>
<h2>📦 Tables</h2><div class="grid">{tbl_html}</div>
<h2>🔗 Relationships</h2>
<table class="rtable"><tr><th>From</th><th>Type</th><th>To</th><th>Verb</th></tr>{rel_html}</table>
<h2>📜 Generated SQL</h2><pre>{self.ddl[:4000]}{'...' if len(self.ddl)>4000 else ''}</pre>
</body></html>"""
        with open(fname, "w", encoding="utf-8") as f:
            f.write(html)
        return fname


print("✅ ReportExporter class defined  (SQL + JSON + HTML).")

✅ ReportExporter class defined  (SQL + JSON + HTML).


---
## 9. Database Executor — MySQL

`DatabaseExecutor` connects to a live MySQL server and runs the generated DDL.

> 💡 This step is **optional** — skip it if you don't have MySQL running.

In [18]:
class DatabaseExecutor:
    """
    Connects to MySQL and executes the generated DDL statements.
    Tracks successes and failures and reports results.
    """

    def __init__(self, host: str, user: str, password: str, database: str):
        self.host, self.user = host, user
        self.password, self.database = password, database
        self.conn = None

    def connect(self, with_db: bool = True) -> bool:
        if not MYSQL_AVAILABLE:
            print("⚠  mysql-connector-python not installed.")
            return False
        try:
            kw = dict(host=self.host, user=self.user, password=self.password)
            if with_db:
                kw["database"] = self.database
            self.conn = mysql.connector.connect(**kw)
            return True
        except Exception as e:
            print(f"✗ Connection failed: {e}")
            return False

    def create_database(self) -> bool:
        if not self.connect(with_db=False):
            return False
        try:
            cur = self.conn.cursor()
            cur.execute(
                f"CREATE DATABASE IF NOT EXISTS `{self.database}` "
                f"CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;"
            )
            self.conn.commit()
            print(f"✓ Database `{self.database}` created/verified.")
            return True
        except Exception as e:
            print(f"✗ {e}")
            return False
        finally:
            self.conn.close()

    def execute_ddl(self, ddl: str) -> dict:
        results = {"success": [], "failed": [], "total": 0}
        if not self.connect():
            return results
        cursor = self.conn.cursor()
        stmts  = [s.strip() for s in ddl.split(";") if s.strip() and not s.strip().startswith("--")]
        for stmt in stmts:
            results["total"] += 1
            try:
                cursor.execute(stmt)
                results["success"].append(stmt[:60])
            except mysql.connector.Error as e:
                results["failed"].append({"stmt": stmt[:60], "error": str(e)})
        self.conn.commit()
        cursor.close()
        self.conn.close()
        return results


print("✅ DatabaseExecutor class defined.")
if not MYSQL_AVAILABLE:
    print("⚠  mysql-connector-python not installed — DB execution will be skipped.")

✅ DatabaseExecutor class defined.


---
## 10. Statistics Helper

A utility function that prints a Rich table summarising the generated schema.

In [19]:
def print_statistics(entities: list, relationships: list, schema: dict):
    """Print a Rich summary table of schema metrics."""
    n_join = sum(1 for e in entities if e.get("is_join_table"))
    n_cols = sum(len(v["columns"]) for v in schema.values())
    n_fk   = sum(len(v["fk_constraints"]) for v in schema.values())
    n_idx  = sum(len(v["indexes"]) for v in schema.values())
    rel_mm = sum(1 for r in relationships if r["rel_type"] == "m:m")
    rel_1m = sum(1 for r in relationships if r["rel_type"] == "1:m")
    rel_11 = sum(1 for r in relationships if r["rel_type"] == "1:1")

    stats = Table(title="[bold]📈  Schema Statistics[/bold]",
                  box=box.ROUNDED, header_style="bold blue", show_header=True)
    stats.add_column("Metric",  style="bold",       min_width=28)
    stats.add_column("Value",   justify="right",    style="cyan",  min_width=8)
    stats.add_column("Details", style="dim",        min_width=30)

    stats.add_row("Total Tables",           str(len(entities)),      f"{n_join} join tables")
    stats.add_row("Total Columns",          str(n_cols),             f"~{n_cols//max(len(entities),1)} per table avg")
    stats.add_row("Foreign Keys",           str(n_fk),               "Referential integrity")
    stats.add_row("Indexes",                str(n_idx),              "Performance optimization")
    stats.add_row("Relationships Total",    str(len(relationships)), "All types combined")
    stats.add_row("  ↳ Many-to-Many (M:M)", str(rel_mm),            "Auto join tables created")
    stats.add_row("  ↳ One-to-Many  (1:M)", str(rel_1m),            "FK on child table")
    stats.add_row("  ↳ One-to-One   (1:1)", str(rel_11),            "Shared PK or UK")

    console.print(stats)


print("✅ print_statistics() helper defined.")

✅ print_statistics() helper defined.


---
## 10. 📝 Interactive Demo — Enter Your Own Text!

**This is the main cell.** Type (or paste) any English description of a system
and the pipeline will automatically:

1. Parse the text with spaCy
2. Run the expert system rules
3. Build a full database schema
4. Render an ER diagram
5. Generate SQL DDL
6. Export `.sql`, `.json`, `.html` files

### ✏️ How to use
- Edit the `USER_TEXT` variable below with your own description
- Run the cell
- Watch the full pipeline execute step by step!

In [20]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║          ✏️  EDIT THIS TEXT — Describe your system in English!          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

USER_TEXT = input(
    """
╔══════════════════════════════════════════════════════════╗
║   🧠  NLP → Database Schema Generator                   ║
║   Enter a description of your system in English.        ║
║   Press ENTER twice when done.                          ║
╚══════════════════════════════════════════════════════════╝

Your text:  """
)

# If nothing entered, fall back to the university example
if not USER_TEXT.strip():
    USER_TEXT = """The university comprises various faculties, each offering several degree programs.
Each program includes multiple subjects taught by qualified professors.
Every professor has a unique Professor ID, full name, and area of expertise.
Students enrolled in the university have a unique Student Number, name, enrollment date, and contact information.
Students can register for multiple subjects each semester, and every subject is identified by a unique Subject Code and title.
Subjects may have prerequisites and are evaluated through exams and assignments.
The campus features several laboratories equipped with specialized equipment.
Students can book laboratory sessions and access research materials.
Additionally, the university maintains a digital archive with theses and publications accessible to all members."""
    print("ℹ  No input detected — using the default University example.")

DATABASE_NAME = "auto_generated_db"

console.print()
console.rule("[bold cyan]📄  Input Text[/bold cyan]")
console.print(Panel(USER_TEXT, title="[dim]Your Input[/dim]", border_style="dim blue"))
print(f"   Characters : {len(USER_TEXT)}")
print(f"   Words      : {len(USER_TEXT.split())}")

ℹ  No input detected — using the default University example.


───────────────────────────────────────────────── 📄  Input Text ──────────────────────────────────────────────────

╭────────────────────────────────────────────────── Your Input ───────────────────────────────────────────────────╮
│ The university comprises various faculties, each offering several degree programs.                              │
│ Each program includes multiple subjects taught by qualified professors.                                         │
│ Every professor has a unique Professor ID, full name, and area of expertise.                                    │
│ Students enrolled in the university have a unique Student Number, name, enrollment date, and contact            │
│ information.                                                                                                    │
│ Students can register for multiple subjects each semester, and every subject is identified by a unique Subject  │
│ Code and title.                                                                                                 │
│ Subjects may have prerequisites and are evaluated through exams and assignments.                                │
│ The campus features several laboratories equipped with specialized equipment.                                   │
│ Students can book laboratory sessions and access research materials.                                            │
│ Additionally, the university maintains a digital archive with theses and publications accessible to all         │
│ members.                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   Characters : 813
   Words      : 112


### Step 1 — NLP Analysis

Parse the input text with spaCy and extract facts.

In [21]:
# ── Step 1: Run spaCy pipeline ───────────────────────────────────────────────
console.rule("[bold]Step 1 · NLP Analysis[/bold]")

nlp_proc = NLPProcessor()
doc      = nlp_proc.process(USER_TEXT)
facts    = nlp_proc.extract_facts(doc)

n_tokens = sum(1 for f in facts if isinstance(f, TokenFact))
n_deps   = sum(1 for f in facts if isinstance(f, DependencyFact))

print(f"\n✅ spaCy analysis complete")
print(f"   Sentences   : {len(list(doc.sents))}")
print(f"   TokenFacts  : {n_tokens}")
print(f"   DepFacts    : {n_deps}")
print(f"   Total facts : {len(facts)}")

console.print()
nlp_proc.show_dependency_tree(doc)

────────────────────────────────────────────── Step 1 · NLP Analysis ──────────────────────────────────────────────

Loading spaCy model 'en_core_web_sm'... ✓

✅ spaCy analysis complete
   Sentences   : 9
   TokenFacts  : 33
   DepFacts    : 9
   Total facts : 42


🌳 Dependency Parse Tree
├── "The university comprises various faculties, each offering several degr..."
│   ├── nsubj        university  (NOUN | NN)
│   ├── ROOT         comprises  (VERB | VBZ)
│   ├── dobj         faculties  (NOUN | NNS)
│   ├── nsubj        each  (PRON | DT)
│   └── dobj         programs  (NOUN | NNS)
├── "Each program includes multiple subjects taught by qualified professors..."
│   ├── nsubj        program  (NOUN | NN)
│   ├── ROOT         includes  (VERB | VBZ)
│   └── dobj         subjects  (NOUN | NNS)
├── "Every professor has a unique Professor ID, full name, and area of expe..."
│   ├── nsubj        professor  (NOUN | NN)
│   ├── ROOT         has  (VERB | VBZ)
│   └── dobj         ID  (PROPN | NNP)
├── "Students enrolled in the university have a unique Student Number, name..."
│   ├── nsubj        Students  (NOUN | NNS)
│   ├── ROOT         enrolled  (VERB | VBD)
│   └── dobj         name  (NOUN | NN)
├── "Students can register for multiple subjects each semester, and every s..."
│   ├── nsubj        Students  (NOUN | NNS)
│   ├── ROOT         register  (VERB | VB)
│   └── nsubjpass    subject  (NOUN | NN)
├── "Subjects may have prerequisites and are evaluated through exams and as..."
│   ├── nsubj        Subjects  (NOUN | NNS)
│   ├── ROOT         have  (VERB | VB)
│   └── dobj         prerequisites  (NOUN | NNS)
├── "The campus features several laboratories equipped with specialized equ..."
│   ├── nsubj        campus  (NOUN | NN)
│   ├── ROOT         features  (VERB | VBZ)
│   └── dobj         laboratories  (NOUN | NNS)
├── "Students can book laboratory sessions and access research materials.
│   "
│   ├── nsubj        Students  (NOUN | NNS)
│   ├── ROOT         book  (VERB | VB)
│   └── dobj         sessions  (NOUN | NNS)
└── "Additionally, the university maintains a digital archive with theses a..."
    ├── nsubj        university  (NOUN | NN)
    ├── ROOT         maintains  (VERB | VBZ)
    └── dobj         archive  (NOUN | NN)

### Step 2 — Expert System Inference

Declare facts and run the 7 rules.

In [22]:
# ── Step 2: Run the expert system engine ────────────────────────────────────
console.rule("[bold]Step 2 · Expert System[/bold]")

engine = UniversityDBEngine()
engine.reset()

for fact in facts:
    engine.declare(fact)

engine.run()

# ── Collect results ───────────────────────────────────────────────────────────
raw_entities, raw_relationships = [], []
for fact in engine.facts.values():
    if isinstance(fact, EntityFact):
        raw_entities.append({
            "name":         fact["name"],
            "attributes":   fact["attributes"],
            "is_join_table":fact.get("is_join_table", False),
            "confidence":   fact.get("confidence", 1.0),
        })
    elif isinstance(fact, RelationshipFact):
        raw_relationships.append(fact)

# Deduplicate entities
seen, entities = set(), []
for e in raw_entities:
    if e["name"] not in seen and len(e["name"]) > 2:
        seen.add(e["name"])
        entities.append(e)

relationships = raw_relationships

print(f"\n✅ Expert system inference complete")
print(f"   Facts in WM     : {len(engine.facts)}")
print(f"   Entities found  : {len(entities)}")
print(f"   Relationships   : {len(relationships)}")

───────────────────────────────────────────── Step 2 · Expert System ──────────────────────────────────────────────


✅ Expert system inference complete
   Facts in WM     : 84
   Entities found  : 33
   Relationships   : 8


### Step 3 — Schema Construction

Build full DDL from the inferred entities and relationships.

In [23]:
# ── Step 3: Build schema ─────────────────────────────────────────────────────
console.rule("[bold]Step 3 · Schema Construction[/bold]")

builder = SchemaBuilder(entities, relationships)
ddl     = builder.generate_ddl(DATABASE_NAME)

n_tables = len(builder.schema)
n_cols   = sum(len(v["columns"]) for v in builder.schema.values())
n_fks    = sum(len(v["fk_constraints"]) for v in builder.schema.values())

print(f"\n✅ Schema built")
print(f"   Tables   : {n_tables}")
print(f"   Columns  : {n_cols}")
print(f"   FK links : {n_fks}")
print(f"   DDL lines: {len(ddl.splitlines())}")

────────────────────────────────────────── Step 3 · Schema Construction ───────────────────────────────────────────


✅ Schema built
   Tables   : 33
   Columns  : 160
   FK links : 8
   DDL lines: 448


### Step 4 — ER Diagram

Visualise the schema as entity cards and a relationship table.

In [24]:
# ── Step 4: ER Diagram ───────────────────────────────────────────────────────
renderer = ERDiagramRenderer(builder.schema, relationships)
renderer.render()

──────────────────────────────────────────  Entity-Relationship Diagram ───────────────────────────────────────────

╭───────────── 📦 Access ─────────────╮ ╭──────────── 📦 Archive ─────────────╮
│                                     │ │                                     │
│   Column       Type           Key   │ │   Column       Type           Key   │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │ │  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│   id           INT UNSIGNED   🔑    │ │   id           INT UNSIGNED   🔑    │
│   created_at   DATETIME             │ │   created_at   DATETIME             │
│   updated_at   DATETIME             │ │   updated_at   DATETIME             │
│                                     │ │                                     │
╰─────────────────────────────────────╯ ╰─────────────────────────────────────╯
╭────────────── 📦 Area ──────────────╮                                        
│                                     │                                        
│   Column       Type           Key   │                                        
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │                                        
│   id           INT UNSIGNED   🔑    │                                        
│   created_at   DATETIME             │                                        
│   updated_at   DATETIME             │                                        
│                                     │                                        
╰─────────────────────────────────────╯

╭─────────── 📦 Assignment ────────────╮ ╭───────────── 📦 Campus ─────────────╮
│                                      │ │                                     │
│   Column        Type           Key   │ │   Column       Type           Key   │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │ │  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│   id            INT UNSIGNED   🔑    │ │   id           INT UNSIGNED   🔑    │
│   title         VARCHAR(200)         │ │   name         VARCHAR(150)         │
│   due_date      DATE                 │ │   address      VARCHAR(300)         │
│   max_score     FLOAT                │ │   area_sqm     VARCHAR(255)         │
│   description   TEXT                 │ │   laboratory   VARCHAR(255)         │
│   created_at    DATETIME             │ │   created_at   DATETIME             │
│   updated_at    DATETIME             │ │   updated_at   DATETIME             │
│                                      │ │                                     │
╰──────────────────────────────────────╯ ╰─────────────────────────────────────╯
╭────────────── 📦 Code ───────────────╮                                        
│                                      │                                        
│   Column       Type           Key    │                                        
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   │                                        
│   id           INT UNSIGNED   🔑     │                                        
│   created_at   DATETIME              │                                        
│   updated_at   DATETIME              │                                        
│                                      │                                        
╰──────────────────────────────────────╯

╭──────────── 📦 Contact ─────────────╮ ╭────────────── 📦 Date ──────────────╮
│                                     │ │                                     │
│   Column       Type           Key   │ │   Column       Type           Key   │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │ │  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│   id           INT UNSIGNED   🔑    │ │   id           INT UNSIGNED   🔑    │
│   created_at   DATETIME             │ │   created_at   DATETIME             │
│   updated_at   DATETIME             │ │   updated_at   DATETIME             │
│                                     │ │                                     │
╰─────────────────────────────────────╯ ╰─────────────────────────────────────╯
╭───────────── 📦 Degree ─────────────╮                                        
│                                     │                                        
│   Column       Type           Key   │                                        
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │                                        
│   id           INT UNSIGNED   🔑    │                                        
│   created_at   DATETIME             │                                        
│   updated_at   DATETIME             │                                        
│                                     │                                        
╰─────────────────────────────────────╯

╭────────────── 📦 Enrollment ──────────────╮ ╭─────────── 📦 Equipment ────────────╮
│                                           │ │                                     │
│   Column       Type           Key         │ │   Column       Type           Key   │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━        │ │  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│   id           INT UNSIGNED   🔑          │ │   id           INT UNSIGNED   🔑    │
│   created_at   DATETIME                   │ │   created_at   DATETIME             │
│   updated_at   DATETIME                   │ │   updated_at   DATETIME             │
│                                           │ │                                     │
╰───────────────────────────────────────────╯ ╰─────────────────────────────────────╯
╭───────────────── 📦 Exam ─────────────────╮                                        
│                                           │                                        
│   Column             Type           Key   │                                        
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │                                        
│   id                 INT UNSIGNED   🔑    │                                        
│   title              VARCHAR(200)         │                                        
│   exam_date          DATE                 │                                        
│   duration_minutes   VARCHAR(255)         │                                        
│   max_score          FLOAT                │                                        
│   created_at         DATETIME             │                                        
│   updated_at         DATETIME             │                                        
│                                           │                                        
╰───────────────────────────────────────────╯

...

╭──────────── 📦 Research ────────────╮ ╭──────────── 📦 Semester ────────────╮
│                                     │ │                                     │
│   Column       Type           Key   │ │   Column       Type           Key   │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │ │  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│   id           INT UNSIGNED   🔑    │ │   id           INT UNSIGNED   🔑    │
│   created_at   DATETIME             │ │   created_at   DATETIME             │
│   updated_at   DATETIME             │ │   updated_at   DATETIME             │
│                                     │ │                                     │
╰─────────────────────────────────────╯ ╰─────────────────────────────────────╯
╭──────────── 📦 Session ─────────────╮                                        
│                                     │                                        
│   Column       Type           Key   │                                        
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │                                        
│   id           INT UNSIGNED   🔑    │                                        
│   id_student   INT UNSIGNED         │                                        
│   created_at   DATETIME             │                                        
│   updated_at   DATETIME             │                                        
│                                     │                                        
╰─────────────────────────────────────╯

╭────────────────── 📦 Student ───────────────────╮ ╭────────────── 🔀 Join ──────────────╮
│                                                 │ │                                     │
│   Column            Type                  Key   │ │   Column       Type           Key   │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │ │  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│   id                INT UNSIGNED          🔑    │ │   id           INT UNSIGNED   🔑    │
│   full_name         VARCHAR(150)                │ │   id_student   INT UNSIGNED         │
│   email             VARCHAR(100) UNIQUE    ◆    │ │   id_session   INT UNSIGNED         │
│   enrollment_date   DATE                        │ │   created_at   DATETIME             │
│   phone             VARCHAR(20)                 │ │   updated_at   DATETIME             │
│   address           VARCHAR(300)                │ │                                     │
│   gpa               FLOAT                       │ ╰─────────────────────────────────────╯
│   created_at        DATETIME                    │                                        
│   updated_at        DATETIME                    │                                        
│                                                 │                                        
╰─────────────────────────────────────────────────╯                                        
╭────────────────── 📦 Subject ───────────────────╮                                        
│                                                 │                                        
│   Column         Type           Key             │                                        
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━            │                                        
│   id             INT UNSIGNED   🔑              │                                        
│   title          VARCHAR(200)                   │                                        
│   description    TEXT                           │                                        
│   credits        FLOAT                          │                                        
│   semester       VARCHAR(255)                   │                                        
│   max_students   VARCHAR(255)                   │                                        
│   prerequisite   VARCHAR(255)                   │                                        
│   id_program     INT UNSIGNED                   │                                        
│   created_at     DATETIME                       │                                        
│   updated_at     DATETIME                       │                                        
│                                                 │                                        
╰─────────────────────────────────────────────────╯

╭──────────────── 🔀 Join ─────────────────╮ ╭──────────────────── 📦 Thesis ─────────────────────╮
│                                          │ │                                                    │
│   Column            Type           Key   │ │   Column            Type                     Key   │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │ │  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│   id                INT UNSIGNED   🔑    │ │   id                INT UNSIGNED             🔑    │
│   id_subject        INT UNSIGNED         │ │   title             VARCHAR(200)                   │
│   id_prerequisite   INT UNSIGNED         │ │   abstract          VARCHAR(255)                   │
│   created_at        DATETIME             │ │   submission_date   DATE                           │
│   updated_at        DATETIME             │ │   status            ENUM('active','inact..         │
│                                          │ │   supervisor_id     INT UNSIGNED             🔗    │
╰──────────────────────────────────────────╯ │   created_at        DATETIME                       │
                                             │   updated_at        DATETIME                       │
                                             │                                                    │
                                             ╰────────────────────────────────────────────────────╯
╭──────────────── 📦 Title ────────────────╮                                                       
│                                          │                                                       
│   Column       Type           Key        │                                                       
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━       │                                                       
│   id           INT UNSIGNED   🔑         │                                                       
│   created_at   DATETIME                  │                                                       
│   updated_at   DATETIME                  │                                                       
│                                          │                                                       
╰──────────────────────────────────────────╯

──────────────────────────────────────────────── 🔗  Relationships ────────────────────────────────────────────────

╭────────────────────┬──────────────┬──────────────────────┬────────────────╮
│ From               │ Cardinality  │ To                   │ Verb           │
├────────────────────┼──────────────┼──────────────────────┼────────────────┤
│ Campus             │  ── 1:M ──   │ Laboratory           │ feature        │
├────────────────────┼──────────────┼──────────────────────┼────────────────┤
│ Student            │  ── M:M ──   │ Session              │ book           │
├────────────────────┼──────────────┼──────────────────────┼────────────────┤
│ Student            │  ── 1:M ──   │ Student_Session      │ links          │
├────────────────────┼──────────────┼──────────────────────┼────────────────┤
│ Session            │  ── 1:M ──   │ Student_Session      │ links          │
├────────────────────┼──────────────┼──────────────────────┼────────────────┤
│ Subject            │  ── M:M ──   │ Prerequisite         │ have           │
├────────────────────┼──────────────┼──────────────────────┼────────────────┤
│ Subject            │  ── 1:M ──   │ Subject_Prerequisite │ links          │
├────────────────────┼──────────────┼──────────────────────┼────────────────┤
│ Prerequisite       │  ── 1:M ──   │ Subject_Prerequisite │ links          │
├────────────────────┼──────────────┼──────────────────────┼────────────────┤
│ Program            │  ── 1:M ──   │ Subject              │ include        │
╰────────────────────┴──────────────┴──────────────────────┴────────────────╯

### Step 5 — Schema Statistics

In [25]:
# ── Step 5: Statistics ───────────────────────────────────────────────────────
print_statistics(entities, relationships, builder.schema)

                           📈  Schema Statistics                            
╭──────────────────────────────┬──────────┬────────────────────────────────╮
│ Metric                       │    Value │ Details                        │
├──────────────────────────────┼──────────┼────────────────────────────────┤
│ Total Tables                 │       33 │ 2 join tables                  │
│ Total Columns                │      160 │ ~4 per table avg               │
│ Foreign Keys                 │        8 │ Referential integrity          │
│ Indexes                      │        5 │ Performance optimization       │
│ Relationships Total          │        8 │ All types combined             │
│   ↳ Many-to-Many (M:M)       │        2 │ Auto join tables created       │
│   ↳ One-to-Many  (1:M)       │        6 │ FK on child table              │
│   ↳ One-to-One   (1:1)       │        0 │ Shared PK or UK                │
╰──────────────────────────────┴──────────┴────────────────────────────────╯

### Step 6 — SQL Preview

Syntax-highlighted preview of the generated DDL.

In [26]:
# ── Step 6: SQL preview (first 70 lines) ────────────────────────────────────
console.rule("[bold]Step 6 · SQL Preview[/bold]")
preview = "\n".join(ddl.splitlines()[:70])
console.print(Syntax(preview, "sql", theme="dracula", line_numbers=True, word_wrap=True))
total_lines = len(ddl.splitlines())
if total_lines > 70:
    print(f"  ... and {total_lines - 70} more lines (see exported .sql file)")

────────────────────────────────────────────── Step 6 · SQL Preview ───────────────────────────────────────────────

   1 -- ═══════════════════════════════════════════════════════                                                    
   2 -- NLP → DB Schema Generator v2.0.0-expert                                                                    
   3 -- Generated : 2026-03-19 04:44:01                                                                            
   4 -- Database  : auto_generated_db                                                                              
   5 -- ═══════════════════════════════════════════════════════                                                    
   6                                                                                                               
   7 SET FOREIGN_KEY_CHECKS = 0;                                                                                   
   8 SET SQL_MODE = 'STRICT_TRANS_TABLES,NO_ZERO_IN_DATE';                                                         
   9                                                                                                               
  10 CREATE DATABASE IF NOT EXISTS `auto_generated_db`                                                             
  11   CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;                                                           
  12                                                                                                               
  13 USE `auto_generated_db`;                                                                                      
  14                                                                                                               
  15 -- ─── Access ────                                                                                            
  16 DROP TABLE IF EXISTS `Access`;                                                                                
  17 CREATE TABLE `Access` (                                                                                       
  18   `id` INT UNSIGNED NOT NULL AUTO_INCREMENT,                                                                  
  19   `created_at` DATETIME DEFAULT CURRENT_TIMESTAMP,                                                            
  20   `updated_at` DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,                                
  21   PRIMARY KEY (`id`)                                                                                          
  22 ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;                                           
  23                                                                                                               
  24                                                                                                               
  25 -- ─── Archive ────                                                                                           
  26 DROP TABLE IF EXISTS `Archive`;                                                                               
  27 CREATE TABLE `Archive` (                                                                                      
  28   `id` INT UNSIGNED NOT NULL AUTO_INCREMENT,                                                                  
  29   `created_at` DATETIME DEFAULT CURRENT_TIMESTAMP,                                                            
  30   `updated_at` DATETIME DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,                                
  31   PRIMARY KEY (`id`)                                                                                          
  32 ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;                                           
  33                                                                                                               
  34                                                                                                               
  35 -- ─── Area ────                       

  ... and 378 more lines (see exported .sql file)


### Step 7 — Export Reports

Save the results as `.sql`, `.json`, and `.html` files.

In [27]:
# ── Step 7: Export ───────────────────────────────────────────────────────────
console.rule("[bold]Step 7 · Export[/bold]")

exporter = ReportExporter(builder.schema, relationships, ddl, DATABASE_NAME)

sql_file  = exporter.export_sql(".")
json_file = exporter.export_json(".")
html_file = exporter.export_html(".")

print(f"\n✅ Export complete!")
print(f"   SQL  → {sql_file}")
print(f"   JSON → {json_file}")
print(f"   HTML → {html_file}")

───────────────────────────────────────────────── Step 7 · Export ─────────────────────────────────────────────────


✅ Export complete!
   SQL  → ./schema_20260319_044855.sql
   JSON → ./schema_20260319_044855.json
   HTML → ./schema_20260319_044855.html


### Step 8 — MySQL Execution (Optional)

> ⚠️ Only run this cell if you have a MySQL server running.
> Fill in your credentials first.

In [ ]:
# ── Step 8: MySQL execution ──────────────────────────────────────────────────
# 🔧 CONFIGURE YOUR CREDENTIALS HERE
MYSQL_HOST     = "localhost"
MYSQL_USER     = "root"
MYSQL_PASSWORD = ""               # ← your MySQL password
MYSQL_DATABASE = DATABASE_NAME

if not MYSQL_AVAILABLE:
    print("⚠  mysql-connector-python not installed. Skipping.")
    print("   Run:  pip install mysql-connector-python")
else:
    executor = DatabaseExecutor(MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DATABASE)
    if executor.create_database():
        results = executor.execute_ddl(ddl)
        ok  = len(results["success"])
        bad = len(results["failed"])
        print(f"\n✅ Execution complete: {ok} OK · {bad} failed")
        if results["failed"]:
            print("\nFailed statements:")
            for f in results["failed"][:5]:
                print(f"  ✗ {f['stmt'][:70]}  →  {f['error']}")

---
## 11. Results & Statistics

Run the cell below after the pipeline finishes for a final summary.

In [28]:
# ── Final summary panel ───────────────────────────────────────────────────────
console.print()
console.print(Panel(
    f"[bold green]✅  Pipeline Complete![/bold green]\n\n"
    f"  Input words    : [cyan]{len(USER_TEXT.split())}[/cyan]\n"
    f"  NLP facts      : [cyan]{len(facts)}[/cyan]  "
      f"({n_tokens} tokens + {n_deps} dependencies)\n"
    f"  Entities       : [cyan]{len(entities)}[/cyan]\n"
    f"  Relationships  : [cyan]{len(relationships)}[/cyan]\n"
    f"  Tables created : [cyan]{len(builder.schema)}[/cyan]\n"
    f"  Columns total  : [cyan]{sum(len(v['columns']) for v in builder.schema.values())}[/cyan]\n"
    f"  DDL lines      : [cyan]{len(ddl.splitlines())}[/cyan]\n"
    f"  Output files   : [dim]schema_*.sql / .json / .html[/dim]",
    title="[bold] 🧠 NLP → DB Schema Generator  —  Summary[/bold]",
    border_style="green",
    padding=(1, 4)
))

╭───────────────────────────────────  🧠 NLP → DB Schema Generator  —  Summary ───────────────────────────────────╮
│                                                                                                                 │
│    ✅  Pipeline Complete!                                                                                       │
│                                                                                                                 │
│      Input words    : 112                                                                                       │
│      NLP facts      : 42  (33 tokens + 9 dependencies)                                                          │
│      Entities       : 33                                                                                        │
│      Relationships  : 8                                                                                         │
│      Tables created : 33                                                                                        │
│      Columns total  : 160                                                                                       │
│      DDL lines      : 448                                                                                       │
│      Output files   : schema_*.sql / .json / .html                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---

## 📄 License 

```
MIT License — feel free to use, modify and distribute.
```


### 📚 References

- [spaCy Documentation](https://spacy.io/usage)
- [Experta (PyKnow fork)](https://github.com/nilp0inter/experta)
- [Rich Terminal UI](https://github.com/Textualize/rich)
- [MySQL Connector/Python](https://dev.mysql.com/doc/connector-python/en/)

---

## 👩‍💻 About the Author

<div align="center">

<img src="https://readme-typing-svg.herokuapp.com?font=Fira+Code&size=22&pause=1000&color=06B6D4&center=true&vCenter=true&width=600&lines=ROZÊRA+XELÎL;Full-Stack+Developer+%26+AI+Student;From+ROJAVA+;Bridging+Human+Language+%26+Machine+Logic" alt="Typing SVG" />

</div>

<br>

<div align="center">

| | |
|:---:|:---|
| 🌍 **Origin** | ROJAVA |
| 💼 **Role** | Full-Stack Developer & AI Student 🤖 |
| 🧠 **Focus** | Bridging the gap between human language and machine logic |
| 💡 | `2 + 2 = 1` ✌🏻 ❤️ 🤍 💛 💚 |

</div>

<br>

<div align="center">

> *"Dedicated to bridging the gap between human language and machine logic."*
>
> 💥✨ **`2 + 2 = 1`** —. ✌🏻❤️🤍💛💚
>
> 🧩⚡ *Code is the language the universe speaks when it wants to evolve.*

</div>

<br>

<div align="center">

![Rojava](https://img.shields.io/badge/From-ROJAVA_-red?style=for-the-badge)
![Role](https://img.shields.io/badge/Full--Stack_Developer-06B6D4?style=for-the-badge&logo=code&logoColor=white)
![AI](https://img.shields.io/badge/AI_Student_🤖-8A2BE2?style=for-the-badge&logo=openai&logoColor=white)
![NLP](https://img.shields.io/badge/NLP_Enthusiast-22c55e?style=for-the-badge&logo=spacy&logoColor=white)

</div>

---

<div align="center">

Made with ❤️ by **ROZÊRA XELÎL** · NLP + Expert Systems + Auto SQL Generation +

*ROJAVA ✌🏻❤️🤍💛💚*

</div>